In [14]:
import pandas as pd
import re
import os

# --- Step 1: Load CSV ---
train_df = pd.read_csv("../dataset/train.csv")
test_df = pd.read_csv("../dataset/test.csv")

In [15]:
# --- Step 2: Find the rows which don't have images in image folder
def remove_bad_images(dataframe, image_folder):
    image_files = set(os.listdir(image_folder))
    missing_images = dataframe[~dataframe['sample_id'].astype(str).apply(lambda x: f"{x}.jpg" in image_files)]
    dataframe = dataframe[dataframe['sample_id'].astype(str).apply(lambda x: f"{x}.jpg" in image_files)]
    dataframe.reset_index(drop=True, inplace=True)
    print(f"Removed {len(missing_images)} rows with missing images.")
    if (not missing_images.empty):
        print("Missing image sample_ids:", missing_images['sample_id'].tolist())
    return missing_images, dataframe

# missing_train, train_df = remove_bad_images(train_df, "../images/train")
# missing_test, test_df = remove_bad_images(test_df, "../images/test")

In [16]:
# --- Step 3: Extract info from catalog_content ---
def parse_catalog(text):
    text = str(text)
    
    # Item Name
    item_name = re.search(r'Item Name:\s*(.*?)(?:\n|$)', text) or re.search(r'Item Name 2:\s*(.*?)(?:\n|$)', text)
    item_name = item_name.group(1).strip() if item_name else "N/A"
    
    # Value & Unit
    value = re.search(r'Value:\s*([\d.]+)', text)
    value = value.group(1).strip() if value else "N/A"
    
    unit = re.search(r'(?m)^Unit:\s*(.*?)(?:\n|$)', text)
    unit = unit.group(1).strip() if unit else "N/A"

    # Combine all bullet points
    bullets = re.findall(r'Bullet Point \d+:\s*(.*?)(?:\n|$)', text)
    description = ' | '.join(bullets).strip() if bullets else "N/A"
    
    return pd.Series([item_name, description, value, unit])

# Apply parser
train_df[['Item Name', 'Description', 'Value', 'Unit']] = train_df['catalog_content'].apply(parse_catalog)
test_df[['Item Name', 'Description', 'Value', 'Unit']] = test_df['catalog_content'].apply(parse_catalog)

In [17]:
# --- Step 4: Clean and normalize units ---
print("Unique units in train set:", train_df['Unit'].nunique())
print("Unique units in test set:", test_df['Unit'].nunique())

Unique units in train set: 92
Unique units in test set: 105


In [18]:
print("Most common units in train set:\n", list(train_df['Unit'].value_counts().items()))
# print("Most common units in test set:\n", test_df['Unit'].value_counts())

Most common units in train set:
 [('Ounce', 40984), ('Count', 17457), ('Fl Oz', 11086), ('ounce', 1960), ('oz', 962), ('None', 940), ('count', 741), ('pound', 158), ('fl oz', 128), ('Oz', 78), ('ct', 46), ('Pound', 44), ('lb', 30), ('Fluid Ounce', 30), ('FL Oz', 28), ('OZ', 24), ('Fl. Oz', 21), ('gram', 20), ('Gram', 19), ('COUNT', 16), ('FL OZ', 15), ('each', 14), ('ounces', 14), ('packs', 11), ('millilitre', 9), ('fluid ounces', 8), ('Each', 7), ('Liters', 7), ('Fluid Ounces', 6), ('fluid ounce(s)', 6), ('gramm', 6), ('Ounces', 6), ('Pack', 6), ('bottle', 6), ('kg', 6), ('Can', 5), ('Bag', 5), ('Fl oz', 4), ('LB', 4), ('Jar', 4), ('per Carton', 3), ('grams', 3), ('Bottle', 3), ('Piece', 3), ('milliliter', 3), ('K-Cups', 3), ('CT', 3), ('pounds', 3), ('fluid ounce', 2), ('per Box', 2), ('Foot', 2), ('Paper Cupcake Liners', 2), ('1', 2), ('product_weight', 2), ('Per Package', 2), ('gr', 2), ('Pouch', 2), ('ml', 2), ('pack', 2), ('', 1), ('-', 1), ('sq ft', 1), ('fl. oz.', 1), ('PACK', 

In [19]:
unit_dict = {
    # --- Ounce / oz family ---
    'ounce': 'ounce',
    'ounces': 'ounce',
    'oz': 'ounce',
    '20 oz.': 'ounce',
    '7,2 oz': 'ounce',
    '1.76 ounces': 'ounce',
    '16 ounces': 'ounce',
    '16.9 oz': 'ounce',
    '1.76 ounce': 'ounce',

    # --- Fluid ounce / fl oz family ---
    'fl oz': 'fluid_ounce',
    'fl. oz': 'fluid_ounce',
    'fl.oz': 'fluid_ounce',
    'fl oz.': 'fluid_ounce',
    'fluid ounce': 'fluid_ounce',
    'fluid ounces': 'fluid_ounce',
    'fluid ounce(s)': 'fluid_ounce',
    'fluid ounces.': 'fluid_ounce',
    'fl_oz': 'fluid_ounce',
    'fl ounce': 'fluid_ounce',
    'fl. oz.': 'fluid_ounce',
    'fluid_ounces': 'fluid_ounce',

    # --- Pound / lb family ---
    'pound': 'lb',
    'pounds': 'lb',
    'lb': 'lb',
    'lbs': 'lb',

    # --- Gram / g family ---
    'gram': 'gram',
    'grams': 'gram',
    'gramm': 'gram',
    'gr': 'gram',
    'g': 'gram',
    'grams(gm)': 'gram',
    'm': 'metre',

    # --- Kilogram / kg family ---
    'kg': 'kilogram',
    'kgs': 'kilogram',

    # --- Millilitre / milliliter / ml family ---
    'millilitre': 'millilitre',
    'millilitres': 'millilitre',
    'milliliter': 'millilitre',
    'milliliters': 'millilitre',
    'mililitro': 'millilitre',
    'millilitro': 'millilitre',
    'ml': 'millilitre',

    # --- Liter / litre family ---
    'liter': 'litre',
    'liters': 'litre',
    'litre': 'litre',
    'litres': 'litre',
    'ltr': 'litre',
    'ltrs': 'litre',
    'l': 'litre',
    'gallon': 'gallon',
    'gallons': 'gallon',

    # --- Count / ct family ---
    'count': 'count',
    'ct': 'count',
    'cts': 'count',
    'count(s)': 'count',
    'unità': 'count',
    'units': 'count',
    '': 'count',
    '-': 'count',
    '---': 'count',
    '....': 'count',
    '1': 'count',
    '2': 'count',
    '24': 'count',
    '8': 'count',
    'ziplock bags': 'count',
    '(pack of 1)': 'count',
    'count / count': 'count',
    'pac': 'count',
    'packet': 'count',
    'cou': 'count',
    '12.54': 'count',

    # --- Each / Pack / Piece family ---
    'each': 'count',
    'ea': 'count',
    'piece': 'count',
    'pack': 'count',
    'packs': 'count',
    'per box': 'count',
    'per carton': 'count',
    'per package': 'count',
    'each / pack: 1': 'count',
    'box': 'count',
    'box/12': 'count',
    'bottle': 'count',
    'bottles': 'count',
    'jar': 'count',
    'jars': 'count',
    'bag': 'count',
    'bucket': 'count',
    'can': 'count',
    'capsule': 'count',
    'capsules': 'count',
    'container': 'count',
    'carton': 'count',
    'k-cups': 'count',
    'paper cupcake liners': 'count',
    'pouch': 'count',
    'product_weight': 'count',
    'tea bags': 'count',
    'kit': 'count',
    'pieces': 'count',
    'sachet': 'count',
    'stück': 'count',
    'sugar substitute': 'count',
    'tin': 'count',
    'unit': 'count',

    # --- Foot / Inch / Area units ---
    'foot': 'foot',
    'feet': 'foot',
    'ft': 'foot',
    'in': 'inch',
    'inch': 'inch',
    'inches': 'inch',
    'sq ft': 'square_foot',
    'sqft': 'square_foot',

    'none': 'na',
}

# Apply normalization
train_df['Unit'] = train_df['Unit'].str.strip().str.lower().replace(unit_dict)
test_df['Unit'] = test_df['Unit'].str.strip().str.lower().replace(unit_dict)

In [20]:
print("Unique units in train set:", train_df['Unit'].nunique())
print("Unique units in test set:", test_df['Unit'].nunique())

print("Most common units in train set:\n", list(train_df['Unit'].value_counts().sort_index().items()))
print("Most common units in test set:\n", list(test_df['Unit'].value_counts().sort_index().items()))

# print("Entries with ambiguous units in train set:\n", list(test_df[test_df['Unit'] == 'unit']['catalog_content'])[0])


Unique units in train set: 12
Unique units in test set: 13
Most common units in train set:
 [('count', 18368), ('fluid_ounce', 11338), ('foot', 2), ('gram', 51), ('inch', 1), ('kilogram', 6), ('lb', 240), ('litre', 8), ('millilitre', 15), ('na', 940), ('ounce', 44029), ('square_foot', 2)]
Most common units in test set:
 [('count', 18359), ('fluid_ounce', 11412), ('foot', 4), ('gallon', 3), ('gram', 36), ('kilogram', 2), ('lb', 221), ('litre', 9), ('metre', 1), ('millilitre', 19), ('na', 1012), ('ounce', 43920), ('square_foot', 2)]


In [21]:
# --- Step 5: Reorder columns so price is last ---
cols = [c for c in train_df.columns if c != 'price'] + ['price']
train_df = train_df[cols]

In [22]:
# --- Step 6: Save final cleaned CSV ---
os.makedirs("../clean_data", exist_ok=True)

# Drop old column
train_df.drop(columns=['catalog_content'], inplace=True)
test_df.drop(columns=['catalog_content'], inplace=True)

train_df.to_csv("../clean_data/train.csv", index=False)
test_df.to_csv("../clean_data/test.csv", index=False)
print("Cleaned data saved")

Cleaned data saved
